# Dataset Simulation Setup

This notebook loads the two CSV datasets in this folder (`Experimental_data_fresh_cell.csv` and `Aggregate.csv`) and reshapes them to match the parameters exposed by the battery SmartShunt and AC multimeter sensors shown in the reference tables.


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

# Base directory = location of this notebook
BASE_DIR = Path().resolve()
BATTERY_CSV = BASE_DIR / "Experimental_data_fresh_cell.csv"
AC_CSV = BASE_DIR / "Aggregate.csv"

print("Using base directory:", BASE_DIR)
print("Battery CSV:", BATTERY_CSV)
print("AC CSV:", AC_CSV)


def load_battery_dataset(path: Path) -> pd.DataFrame:
    """Load Experimental_data_fresh_cell.csv and shape it like the
    Victron SmartShunt-style battery monitor output.
    """
    df = pd.read_csv(path)

    # Expect at least these columns
    required = {"Time", "Current", "Voltage"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Battery CSV missing required columns: {missing}")

    battery_power_w = df["Voltage"] * df["Current"]

    # Build a DataFrame with columns that mirror the parameter table
    out = pd.DataFrame({
        "Time (s)": df["Time"],
        "Battery Voltage (V)": df["Voltage"],
        "Battery Current (A)": df["Current"],
        "Battery Power (W)": battery_power_w,
        # The following are placeholders so the structure matches the spec;
        # you can later fill them with a proper battery model if needed.
        "State of Charge (SoC %)": np.nan,
        "Consumed Amp-hours (Ah)": np.nan,
        "Time-to-Go (h)": np.nan,
        "Alarm / Status Flags": np.nan,
        # Extra but useful channel from the raw file
        "Temperature (C)": df.get("Temperature", np.nan),
    })

    return out


def load_ac_dataset(path: Path, phase: int = 1) -> pd.DataFrame:
    """Load Aggregate.csv and shape it like a PZEM-004T-style AC multimeter.

    Parameters
    ----------
    path : Path
        Path to Aggregate.csv.
    phase : int
        Which phase to map to the generic AC parameters (1, 2, or 3).
    """
    if phase not in (1, 2, 3):
        raise ValueError("phase must be 1, 2, or 3")

    df = pd.read_csv(path)

    v_col = f"Voltage{phase}"
    i_col = f"Current{phase}"
    p_col = f"Real Power{phase}"
    pf_col = f"PF{phase}"

    required = {"Date", "Time", "Frequency", v_col, i_col, p_col, pf_col}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"AC CSV missing required columns: {missing}")

    # Approximate cumulative active energy (kWh) assuming 1 s sampling
    dt_seconds = 1.0
    step_energy_kwh = df[p_col] * dt_seconds / 3600.0 / 1000.0  # W -> kW, then *h
    active_energy_kwh = step_energy_kwh.cumsum()

    out = pd.DataFrame({
        "Date": df["Date"],
        "Time": df["Time"],
        "AC Voltage (V)": df[v_col],
        "AC Current (A)": df[i_col],
        "Active Power (W)": df[p_col],
        "Active Energy (kWh)": active_energy_kwh,
        "Frequency (Hz)": df["Frequency"],
        "Power Factor": df[pf_col],
    })

    return out


# Load both sensor-shaped datasets
battery_df = load_battery_dataset(BATTERY_CSV)
ac_df = load_ac_dataset(AC_CSV, phase=1)

print("Battery-style dataset (first 5 rows):")
display(battery_df.head())

print("AC-meter-style dataset (first 5 rows):")
display(ac_df.head())

# Optionally, save the shaped datasets for use elsewhere
battery_out = BASE_DIR / "battery_sensor_like.csv"
ac_out = BASE_DIR / "ac_sensor_like.csv"

battery_df.to_csv(battery_out, index=False)
ac_df.to_csv(ac_out, index=False)

print("Saved:")
print(" -", battery_out)
print(" -", ac_out)


Using base directory: C:\Users\tayya\Desktop\workares\hatpebble dataset gen
Battery CSV: C:\Users\tayya\Desktop\workares\hatpebble dataset gen\Experimental_data_fresh_cell.csv
AC CSV: C:\Users\tayya\Desktop\workares\hatpebble dataset gen\Aggregate.csv
Battery-style dataset (first 5 rows):


C:\Users\tayya\AppData\Local\Temp\ipykernel_43420\3823247978.py:61: DtypeWarning: Columns (0: Date, 1: Time) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


,Time (s),Battery Voltage (V),Battery Current (A),Battery Power (W),State of Charge (SoC %),Consumed Amp-hours (Ah),Time-to-Go (h),Alarm / Status Flags,Temperature (C)
0,0.000000,2.999607,2.158704,6.475263,NaN,NaN,NaN,NaN,26.384377
1,1.000000,2.999407,2.287674,6.861664,NaN,NaN,NaN,NaN,26.227879
2,2.000000,2.999757,2.228280,6.684298,NaN,NaN,NaN,NaN,26.449251
3,2.999992,2.999857,2.224886,6.674339,NaN,NaN,NaN,NaN,26.277494
4,4.000000,2.999958,2.134946,6.404748,NaN,NaN,NaN,NaN,26.380539


AC-meter-style dataset (first 5 rows):


,Date,Time,AC Voltage (V),AC Current (A),Active Power (W),Active Energy (kWh),Frequency (Hz),Power Factor
0,10/11/2023,11:55:00 PM,243.16,23.1,5300.0,0.001472,50.035,0.95
1,10/11/2023,11:55:01 PM,243.16,23.0,5300.0,0.002944,50.035,0.95
2,10/11/2023,11:55:02 PM,243.22,23.0,5300.0,0.004417,50.036,0.95
3,10/11/2023,11:55:03 PM,243.20,23.1,5300.0,0.005889,50.036,0.95
4,10/11/2023,11:55:04 PM,243.22,23.1,5300.0,0.007361,50.037,0.95


Saved:
 - C:\Users\tayya\Desktop\workares\hatpebble dataset gen\battery_sensor_like.csv
 - C:\Users\tayya\Desktop\workares\hatpebble dataset gen\ac_sensor_like.csv


In [4]:
import json
import time

# Simulate SmartShunt-style battery monitor JSON stream
# Each row of `battery_df` is emitted as if it were a
# real-time sample from the battery monitor.

SIM_DELAY_SECONDS = 0.5  # adjust to taste
N_SAMPLES = 20           # number of rows to stream in this demo

for _, row in battery_df.head(N_SAMPLES).iterrows():
    # Use .at[] to make it clear we are accessing scalar values,
    # which avoids type-checker confusion about possible Series.
    soc = row.at["State of Charge (SoC %)"]
    ah = row.at["Consumed Amp-hours (Ah)"]
    tgo = row.at["Time-to-Go (h)"]
    flags = row.at["Alarm / Status Flags"]
    temp = row.at["Temperature (C)"]

    payload = {
        "Battery Voltage": float(row.at["Battery Voltage (V)"]),
        "Battery Current": float(row.at["Battery Current (A)"]),
        "Battery Power": float(row.at["Battery Power (W)"]),
        "State of Charge (SoC)": None if pd.isna(soc) else float(soc),
        "Consumed Amp-hours": None if pd.isna(ah) else float(ah),
        "Time-to-Go": None if pd.isna(tgo) else float(tgo),
        "Alarm / Status Flags": None if pd.isna(flags) else flags,
        "Temperature": None if pd.isna(temp) else float(temp),
        "Time": float(row.at["Time (s)"]),
    }

    print(json.dumps(payload))
    time.sleep(SIM_DELAY_SECONDS)

print("Battery JSON stream simulation complete.")

{"Battery Voltage": 2.9996068, "Battery Current": 2.1587039, "Battery Power": 6.47526289762652, "State of Charge (SoC)": null, "Consumed Amp-hours": null, "Time-to-Go": null, "Alarm / Status Flags": null, "Temperature": 26.384377, "Time": 0.0}
{"Battery Voltage": 2.9994066, "Battery Current": 2.2876738, "Battery Power": 6.861663894367079, "State of Charge (SoC)": null, "Consumed Amp-hours": null, "Time-to-Go": null, "Alarm / Status Flags": null, "Temperature": 26.227879, "Time": 1.0}
{"Battery Voltage": 2.9997571, "Battery Current": 2.2282798, "Battery Power": 6.684298150836581, "State of Charge (SoC)": null, "Consumed Amp-hours": null, "Time-to-Go": null, "Alarm / Status Flags": null, "Temperature": 26.449251, "Time": 2.0}
{"Battery Voltage": 2.9998572, "Battery Current": 2.2248857, "Battery Power": 6.674339386322041, "State of Charge (SoC)": null, "Consumed Amp-hours": null, "Time-to-Go": null, "Alarm / Status Flags": null, "Temperature": 26.277494, "Time": 2.99999237060547}
{"Batter

In [2]:
import json
import time

# Simulate PZEM-style AC multimeter JSON stream
# Each row from `ac_df` is emitted as if coming from the sensor.

SIM_DELAY_SECONDS = 0.5  # adjust to taste
N_SAMPLES = 20           # number of rows to stream in this demo

for _, row in ac_df.head(N_SAMPLES).iterrows():
    # Use .at[] to make scalar access explicit for the type checker
    v = row.at["AC Voltage (V)"]
    i = row.at["AC Current (A)"]
    p = row.at["Active Power (W)"]
    e = row.at["Active Energy (kWh)"]
    f = row.at["Frequency (Hz)"]
    pf = row.at["Power Factor"]
    date = row.at["Date"]
    t = row.at["Time"]

    payload = {
        "AC Voltage": float(v),
        "AC Current": float(i),
        "Active Power": float(p),
        "Active Energy": float(e),
        "Frequency": float(f),
        "Power Factor": float(pf),
        "Date": str(date),
        "Time": str(t),
    }

    print(json.dumps(payload))
    time.sleep(SIM_DELAY_SECONDS)

print("AC JSON stream simulation complete.")

{"AC Voltage": 243.16, "AC Current": 23.1, "Active Power": 5300.0, "Active Energy": 0.0014722222222222224, "Frequency": 50.035, "Power Factor": 0.95, "Date": "10/11/2023", "Time": "11:55:00 PM"}
{"AC Voltage": 243.16, "AC Current": 23.0, "Active Power": 5300.0, "Active Energy": 0.002944444444444445, "Frequency": 50.035, "Power Factor": 0.95, "Date": "10/11/2023", "Time": "11:55:01 PM"}
{"AC Voltage": 243.22, "AC Current": 23.0, "Active Power": 5300.0, "Active Energy": 0.004416666666666668, "Frequency": 50.036, "Power Factor": 0.95, "Date": "10/11/2023", "Time": "11:55:02 PM"}
{"AC Voltage": 243.2, "AC Current": 23.1, "Active Power": 5300.0, "Active Energy": 0.00588888888888889, "Frequency": 50.036, "Power Factor": 0.95, "Date": "10/11/2023", "Time": "11:55:03 PM"}
{"AC Voltage": 243.22, "AC Current": 23.1, "Active Power": 5300.0, "Active Energy": 0.007361111111111112, "Frequency": 50.037, "Power Factor": 0.95, "Date": "10/11/2023", "Time": "11:55:04 PM"}
{"AC Voltage": 243.18, "AC Curr